# Oracle Discounted Sum-of-Rewards (12 Policies)

**Date:** 2026-03-26  
**Purpose:** Compute oracle value as **sum of per-step rewards** for all 12 policies from v0.2.5.14g.

**Key difference from previous oracle:** Runs each policy for the full 60-step horizon with **no early termination on success**. This lets us compute `sum_t r_t` where `r_t = 1.0 if cube_z > 0.84` at each timestep — the same quantity that `compute_ope_hard()` measures on synthetic trajectories.

Previous oracle: binary 0/1 per episode (success rate), with early termination.  
New oracle: total timesteps the cube is lifted, over full 60-step horizon.  

**Recording semantics:** Records **pre-step `obs`** as latents — matching the semantics of
synthetic trajectories from the diffuser (trained on pre-step data from `collect.py`).
With no early termination, pre-step recording captures all states correctly (the known
recording bug only loses the success state under early termination).

Rollouts are saved to `rollouts/oracle_full_horizon/{policy_name}/` for reuse.

In [ ]:
import sys, os, json, time
import numpy as np
import h5py
from copy import deepcopy
from pathlib import Path

PROJECT_ROOT = Path("/home1/reishuen/latent_sope")
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "third_party" / "robomimic"))

from latent_sope.robomimic_interface.checkpoints import (
    load_checkpoint, build_rollout_policy_from_checkpoint, build_env_from_checkpoint,
)
from latent_sope.eval.reward_model import LiftRewardFn, make_lift_encoder

# Constants
CKPT_BASE = PROJECT_ROOT / "third_party/robomimic/diffusion_policy_trained_models"
ROLLOUT_SAVE_DIR = PROJECT_ROOT / "rollouts" / "oracle_full_horizon"
OUTPUT_JSON = PROJECT_ROOT / "results/2026-03-26/oracle_sum_rewards_12policies.json"
OBS_KEYS = ["object", "robot0_eef_pos", "robot0_eef_quat", "robot0_gripper_qpos"]

NUM_ROLLOUTS = 50
HORIZON = 60
GAMMA = 1.0
DEVICE = "cuda"
LIFT_THRESHOLD = 0.84
CUBE_Z_INDEX = 2

TARGET_POLICIES = [
    {"name": "50demos_epoch10",  "dir": "lift_diffusion_50demos/20260311134204",  "ckpt": "models/model_epoch_10.pth"},
    {"name": "10demos_epoch10",  "dir": "lift_diffusion_10demos/20260311115828",  "ckpt": "models/model_epoch_10.pth"},
    {"name": "200demos_epoch10", "dir": "lift_diffusion_200demos/20260311141036", "ckpt": "models/model_epoch_10.pth"},
    {"name": "200demos_epoch20", "dir": "lift_diffusion_200demos/20260311141036", "ckpt": "models/model_epoch_20.pth"},
    {"name": "100demos_epoch20", "dir": "lift_diffusion_100demos/20260311135551", "ckpt": "models/model_epoch_20.pth"},
    {"name": "test_checkpoint",  "dir": "test/20260309132349",                   "ckpt": "last.pth"},
    {"name": "50demos_epoch20",  "dir": "lift_diffusion_50demos/20260311134204",  "ckpt": "models/model_epoch_20.pth"},
    {"name": "10demos_epoch30",  "dir": "lift_diffusion_10demos/20260311115828",  "ckpt": "models/model_epoch_30.pth"},
    {"name": "100demos_epoch30", "dir": "lift_diffusion_100demos/20260311135551", "ckpt": "models/model_epoch_30.pth"},
    {"name": "50demos_epoch30",  "dir": "lift_diffusion_50demos/20260311134204",  "ckpt": "models/model_epoch_30.pth"},
    {"name": "50demos_epoch40",  "dir": "lift_diffusion_50demos/20260311134204",  "ckpt": "models/model_epoch_40.pth"},
    {"name": "200demos_epoch40", "dir": "lift_diffusion_200demos/20260311141036", "ckpt": "models/model_epoch_40.pth"},
]

reward_fn = LiftRewardFn()
encoder = make_lift_encoder()

print(f"Reward fn: {reward_fn}")
print(f"Config: {NUM_ROLLOUTS} rollouts, horizon={HORIZON}, gamma={GAMMA}, no early termination")
print(f"{len(TARGET_POLICIES)} policies to evaluate")
print(f"Saving rollouts to: {ROLLOUT_SAVE_DIR}")

In [ ]:
# ── Full-horizon rollout + save (no early termination) ──

def full_horizon_rollout(policy, env, horizon, obs_keys):
    """Run a single rollout for the full horizon without early termination.
    
    Records pre-step obs as latents — matching the semantics of synthetic
    trajectories from the diffuser (trained on pre-step data from collect.py).
    With no early termination, pre-step recording captures all relevant states.
    
    Returns:
        latents: (horizon, state_dim) array of pre-step observations
        actions: (horizon, action_dim) array
        rewards_hard: (horizon,) per-step hard reward (cube_z > 0.84)
        env_reward_total: float, cumulative env reward
        ever_success: bool, whether cube was ever lifted
    """
    policy.start_episode()
    obs = env.reset()
    state_dict = env.get_state()
    obs = env.reset_to(state_dict)

    latents_list = []
    actions_list = []
    ever_success = False
    env_reward_total = 0.0

    for step_i in range(horizon):
        # Record pre-step obs as latent (matches synthetic trajectory semantics)
        latent = np.concatenate([np.asarray(obs[k]) for k in obs_keys])
        latents_list.append(latent)

        # Act
        act = policy(ob=obs)
        actions_list.append(np.asarray(act))

        # Step — do NOT break on success/done
        next_obs, reward, done, info = env.step(act)
        env_reward_total += reward
        if env.is_success()["task"]:
            ever_success = True

        # If env says done (internal horizon), reset to continue
        # This shouldn't happen if env horizon > our horizon (60 << 400)
        if done:
            obs = env.reset()
            state_dict = env.get_state()
            obs = env.reset_to(state_dict)
        else:
            obs = deepcopy(next_obs)

    latents = np.stack(latents_list, axis=0).astype(np.float32)  # (H, state_dim)
    actions = np.stack(actions_list, axis=0).astype(np.float32)  # (H, action_dim)

    # Compute per-step hard reward from pre-step cube_z
    # This matches compute_ope_hard() which operates on synthetic states
    cube_z = latents[:, CUBE_Z_INDEX]
    rewards_hard = (cube_z > LIFT_THRESHOLD).astype(np.float32)

    return latents, actions, rewards_hard, env_reward_total, ever_success


def save_rollout_h5(path, latents, actions, rewards_hard, env_reward_total, ever_success, horizon):
    """Save a single rollout to .h5."""
    with h5py.File(path, "w") as f:
        f.create_dataset("latents", data=latents, compression="gzip")
        f.create_dataset("actions", data=actions, compression="gzip")
        f.create_dataset("rewards_hard", data=rewards_hard)
        f.attrs["env_reward_total"] = env_reward_total
        f.attrs["success"] = int(ever_success)
        f.attrs["horizon"] = horizon
        f.attrs["sum_hard_reward"] = float(rewards_hard.sum())


# ── Main loop ──
results = {}
t0_all = time.time()

for i, pol in enumerate(TARGET_POLICIES):
    name = pol["name"]
    run_dir = CKPT_BASE / pol["dir"]

    print(f"\n[{i+1}/{len(TARGET_POLICIES)}] {name}")
    t0 = time.time()

    # Load policy + env
    ckpt = load_checkpoint(run_dir, ckpt_path=Path(pol["ckpt"]))
    policy = build_rollout_policy_from_checkpoint(ckpt, device=DEVICE, verbose=False)
    env = build_env_from_checkpoint(ckpt, render=False, render_offscreen=False, verbose=False)

    # Create save dir
    save_dir = ROLLOUT_SAVE_DIR / name
    save_dir.mkdir(parents=True, exist_ok=True)

    per_rollout_returns = []
    n_success = 0

    for j in range(NUM_ROLLOUTS):
        latents, actions, rewards_hard, env_rew, success = full_horizon_rollout(
            policy, env, HORIZON, OBS_KEYS,
        )

        # Save to disk
        save_rollout_h5(
            save_dir / f"rollout_{j:04d}.h5",
            latents, actions, rewards_hard, env_rew, success, HORIZON,
        )

        # Compute discounted return (matches compute_ope_hard formula exactly):
        # ret = sum_t gamma^t * (cube_z_t > 0.84)
        discounts = GAMMA ** np.arange(HORIZON)
        ret = float((discounts * rewards_hard).sum())
        per_rollout_returns.append(ret)
        if success:
            n_success += 1

        if (j + 1) % 10 == 0:
            running_mean = np.mean(per_rollout_returns)
            print(f"  [{j+1}/{NUM_ROLLOUTS}] running mean={running_mean:.2f}, "
                  f"SR={n_success/(j+1)*100:.0f}%")

    returns = np.array(per_rollout_returns, dtype=np.float32)
    elapsed = time.time() - t0

    results[name] = {
        "mean_return": float(returns.mean()),
        "std_return": float(returns.std()),
        "per_rollout_returns": returns.tolist(),
        "gamma": GAMMA,
        "horizon": HORIZON,
        "num_rollouts": NUM_ROLLOUTS,
        "success_rate": n_success / NUM_ROLLOUTS,
    }

    print(f"  DONE: mean={returns.mean():.2f} +/- {returns.std():.2f}  "
          f"SR={n_success/NUM_ROLLOUTS*100:.0f}%  "
          f"(min={returns.min():.1f}, max={returns.max():.1f})  [{elapsed:.0f}s]")

total_time = time.time() - t0_all
print(f"\nTotal time: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"Rollouts saved to: {ROLLOUT_SAVE_DIR}")

In [ ]:
# ── Save results JSON ──
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
with open(OUTPUT_JSON, "w") as f:
    json.dump(results, f, indent=2)
print(f"Saved: {OUTPUT_JSON}")

In [ ]:
# ── Comparison: old oracle (SR) vs new oracle (sum rewards) ──
OLD_ORACLE = PROJECT_ROOT / "results/2026-03-12/oracle_eval_all_checkpoints.json"
with open(OLD_ORACLE, "r") as f:
    old_oracle = json.load(f)

def get_old_sr(name):
    if name == "test_checkpoint":
        with open(CKPT_BASE / "test/20260309132349/oracle_50.json", "r") as f:
            return json.load(f)["mean_return"]
    elif name in old_oracle:
        return old_oracle[name]["mean_return"]
    return float("nan")

print("=" * 95)
print("COMPARISON: Old Oracle (SR, early-term) vs New Oracle (sum-of-rewards, full horizon)")
print("=" * 95)
print(f"\n{'Policy':<22} {'Old SR':>8} {'New SR':>8} {'Sum Rew':>10} {'Std':>8} {'Min':>6} {'Max':>6}")
print("-" * 75)

for pol in sorted(TARGET_POLICIES, key=lambda p: results[p["name"]]["mean_return"]):
    name = pol["name"]
    new = results[name]
    old_sr = get_old_sr(name)
    returns = np.array(new["per_rollout_returns"])
    print(f"{name:<22} {old_sr:>7.0%} {new['success_rate']:>7.0%} "
          f"{new['mean_return']:>10.2f} {new['std_return']:>8.2f} "
          f"{returns.min():>6.1f} {returns.max():>6.1f}")

print(f"\nSum Rew = total timesteps with cube_z > 0.84 over {HORIZON}-step horizon (gamma={GAMMA})")

# Rank comparison
print(f"\n{'Policy':<22} {'Old SR':>8} {'Sum/T':>8} {'Rank Old':>10} {'Rank New':>10}")
print("-" * 65)
policy_names = [p["name"] for p in TARGET_POLICIES]
old_vals = np.array([get_old_sr(n) for n in policy_names])
new_vals = np.array([results[n]["mean_return"] for n in policy_names])
old_ranks = np.argsort(np.argsort(-old_vals)) + 1
new_ranks = np.argsort(np.argsort(-new_vals)) + 1

from scipy.stats import spearmanr
rho, p = spearmanr(old_vals, new_vals)

for j, name in enumerate(policy_names):
    print(f"{name:<22} {old_vals[j]:>7.0%} {new_vals[j]/HORIZON:>8.3f} "
          f"{old_ranks[j]:>10d} {new_ranks[j]:>10d}")

print(f"\nSpearman rho (old SR vs new sum-reward): {rho:+.3f} (p={p:.4f})")
print(f"If rho ~ 1.0, rankings are preserved and sum-of-rewards adds granularity.")